In [ ]:
#!/usr/bin/env python3
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ======================
# Inputs
# ======================
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"

# GRO-seq normalized readcounts file(s)
# Accepts either:
#   (A) 4 cols with header: chr pos count normalized_count   -> uses normalized_count
#   (B) 3 cols (no header): chr pos value                    -> uses value
GRO_DIR  = "/rhome/zli529/lab/SRA_toolkit/RNAseq_7stages/processed_data/LT"
GRO_GLOB = "LT_coverage.txt"  # wildcard OK

# Outputs
out_dir        = GRO_DIR
plot_mean_png  = os.path.join(out_dir, "GROseq_genes_vs_intergenic_mean.png")
plot_wd_png    = os.path.join(out_dir, "GROseq_genes_vs_intergenic_weighted_density.png")
profiles_tsv   = os.path.join(out_dir, "GROseq_genes_vs_intergenic_profiles.tsv")
wd_tsv         = os.path.join(out_dir, "GROseq_genes_vs_intergenic_weighted_density.tsv")

# ======================
# Parameters
# ======================
UP = 1000          # bp upstream of TSS
DOWN = 1000        # bp downstream of TTS
BINS_BODY = 1000   # bins for normalized gene body
SMOOTH_SD_BP = 25  # Gaussian smoothing SD (bp) for densities; set 0 to disable
CHUNK_SIZE = 2_000_000  # lines per chunk when reading GRO-seq

# ======================
# Helpers
# ======================
def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

def to_prob_density(y):
    y = np.asarray(y, dtype=float)
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def detect_header(path):
    with open(path, "r") as f:
        for line in f:
            if not line.strip() or line.lstrip().startswith("#"):
                continue
            toks = line.strip().split()
            return any(t.lower() in {"chr", "pos", "count", "normalized_count"} for t in toks)
    return False

# ======================
# Load genes / negatives
# ======================
# mRNA: chr start end gene_id strand  → TSS=start & TTS=end on '+', reversed on '-'
mRNA = pd.read_csv(
    mRNA_file, sep=r"\s+|,|\t", header=None,
    names=["chr","start","end","gene_id","strand"],
    dtype={"chr":"string","start":"int64","end":"int64","gene_id":"string","strand":"string"},
    engine="python"
)
mRNA["tss"] = np.where(mRNA["strand"] == "+", mRNA["start"], mRNA["end"])
mRNA["tts"] = np.where(mRNA["strand"] == "+", mRNA["end"],   mRNA["start"])

neg = pd.read_csv(
    neg_file, sep=r"\s+|,|\t", header=None,
    names=["chr","start","end","id","strand"],
    dtype={"chr":"string","start":"int64","end":"int64","id":"string","strand":"string"},
    engine="python"
)
# make a synthetic "gene-like" window for negatives centered at midpoint
neg["center"] = ((neg["start"] + neg["end"]) // 2).astype("int64")

# ======================
# Build GRO-seq coverage dict (chr -> {pos -> value})
# ======================
files = sorted(glob.glob(os.path.join(GRO_DIR, GRO_GLOB)))
assert files, f"No files matched {os.path.join(GRO_DIR, GRO_GLOB)}"

cov_by_chr = {}  # chr -> Series(index=pos int64, values=float)

for path in files:
    has_hdr = detect_header(path)
    if has_hdr:
        # header: chr pos count normalized_count (take chr, pos, normalized_count)
        usecols = [0, 1, 3]
        for chunk in pd.read_csv(
            path, sep=r"\s+", header=0, usecols=usecols, engine="c",
            chunksize=CHUNK_SIZE, dtype={0:"string",1:"int64",3:"float64"}, on_bad_lines="skip"
        ):
            chunk.columns = ["chr","pos","val"]
            if chunk.empty:
                continue
            chunk = chunk.dropna()
            if chunk.empty: continue
            chunk["pos"] = chunk["pos"].astype(np.int64, copy=False)
            grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["val"].sum()
            for chrom, sub in grp.groupby("chr", sort=False):
                s_new = pd.Series(sub["val"].to_numpy(), index=sub["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new
    else:
        # no header: chr pos value  (third col may have ';...' after number)
        for chunk in pd.read_csv(
            path, sep=r"\s+", header=None, names=["chr","pos","raw"], usecols=[0,1,2],
            engine="c", chunksize=CHUNK_SIZE, dtype={"chr":"string","pos":"Int64","raw":"string"},
            na_filter=False, on_bad_lines="skip"
        ):
            raw = chunk["raw"].str.replace(r";.*$", "", regex=True)
            sub = pd.DataFrame({
                "chr": chunk["chr"].astype("string"),
                "pos": pd.to_numeric(chunk["pos"], errors="coerce"),
                "val": pd.to_numeric(raw, errors="coerce")
            }).dropna()
            if sub.empty: continue
            sub["pos"] = sub["pos"].astype(np.int64, copy=False)
            grp = sub.groupby(["chr","pos"], as_index=False, sort=False)["val"].sum()
            for chrom, g in grp.groupby("chr", sort=False):
                s_new = pd.Series(g["val"].to_numpy(), index=g["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new

# tidy
for k, s in list(cov_by_chr.items()):
    s.index = s.index.astype(np.int64, copy=False)
    cov_by_chr[k] = s.sort_index()

# ======================
# Meta-gene profile builders
# ======================
def gene_profile(row, cov_by_chr, up=UP, down=DOWN, bins_body=BINS_BODY):
    chrom = row["chr"]
    s = cov_by_chr.get(chrom)
    if s is None or s.empty:
        return None

    start, end, strand = int(row["start"]), int(row["end"]), row["strand"]
    tss = start if strand == "+" else end
    tts = end   if strand == "+" else start

    region_start = tss - up if strand == "+" else tts - up
    region_end   = tts + down if strand == "+" else tss + down  # exclusive
    length = region_end - region_start
    if length <= 0:
        return None

    # pull coverage at each bp in region (missing -> 0)
    idx = np.arange(region_start, region_end, dtype=np.int64)
    vals = s.reindex(idx, fill_value=0.0).to_numpy()

    # make orientation from TSS->TTS
    if strand == "-":
        vals = vals[::-1]

    # split promoter/body/downstream
    prom = vals[:up]
    body = vals[up: length - down]
    downv = vals[length - down:]

    # normalize body to fixed bins
    if body.size > 1:
        body_x = np.linspace(0, body.size - 1, bins_body)
        body_norm = np.interp(body_x, np.arange(body.size), body)
    else:
        body_norm = np.zeros(bins_body, dtype=float)

    return np.concatenate([prom, body_norm, downv])

def neg_profile(row, cov_by_chr, up=UP, down=DOWN, bins_body=BINS_BODY):
    chrom = row["chr"]
    s = cov_by_chr.get(chrom)
    if s is None or s.empty:
        return None
    center = int(row["center"])
    region_start = center - up
    region_end   = center + bins_body + down   # exclusive
    idx = np.arange(region_start, region_end, dtype=np.int64)
    vals = s.reindex(idx, fill_value=0.0).to_numpy()
    return vals

def average_profiles(df, cov_by_chr, fn):
    profs = []
    for _, row in df.iterrows():
        p = fn(row, cov_by_chr)
        if p is not None:
            profs.append(p)
    if not profs:
        return np.zeros(UP + BINS_BODY + DOWN, dtype=float), 0
    arr = np.vstack(profs)
    return arr.mean(axis=0), arr.shape[0]

# ======================
# Compute mean profiles
# ======================
mean_gene, n_gene = average_profiles(mRNA, cov_by_chr, gene_profile)
mean_neg,  n_neg  = average_profiles(neg,  cov_by_chr, neg_profile)

# Save raw mean profiles
x_axis = np.arange(-UP, BINS_BODY + DOWN, dtype=int)
pd.DataFrame({
    "pos": x_axis,
    "mean_GRO_gene": mean_gene,
    "mean_GRO_intergenic": mean_neg,
    "n_gene_windows": np.full_like(x_axis, n_gene),
    "n_intergenic_windows": np.full_like(x_axis, n_neg),
}).to_csv(profiles_tsv, sep="\t", index=False)
print(f"Wrote: {profiles_tsv}")

# ======================
# Plot mean (absolute signal)
# ======================
plt.figure(figsize=(12, 5))
plt.plot(x_axis, mean_gene, label="genes", lw=2)
plt.plot(x_axis, mean_neg,  label="intergenic", lw=2)
plt.axvspan(-UP, 0, color="green", alpha=0.15, label="TSS −1000 bp")
plt.axvspan(BINS_BODY, BINS_BODY + DOWN, color="red", alpha=0.15, label="TTS +1000 bp")
plt.xticks([-UP, 0, BINS_BODY, BINS_BODY + DOWN], ["-1000", "TSS", "TTS", "+1000"])
plt.xlabel("Position (bp / normalized body)")
plt.ylabel("Mean GRO-seq (normalized)")
plt.title("GRO-seq coverage across genes")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(plot_mean_png, dpi=300)
plt.show()
print(f"Wrote: {plot_mean_png}")

# ======================
# Weighted density (shape × magnitude)
# ======================
dens_gene = to_prob_density(mean_gene)
dens_neg  = to_prob_density(mean_neg)

# optional smoothing (then renormalize to keep area = 1)
if SMOOTH_SD_BP > 0:
    dens_gene = to_prob_density(gaussian_smooth(dens_gene, sd_bp=SMOOTH_SD_BP))
    dens_neg  = to_prob_density(gaussian_smooth(dens_neg,  sd_bp=SMOOTH_SD_BP))

# masses reflect absolute signal over the whole meta-gene region
mass_gene = float(np.nansum(mean_gene))
mass_neg  = float(np.nansum(mean_neg))

# scale densities by relative mass (largest mass = 1)
max_mass = max(mass_gene, mass_neg, 1e-12)
wd_gene = dens_gene * (mass_gene / max_mass)
wd_neg  = dens_neg  * (mass_neg  / max_mass)

# Save weighted density (raw; not display-scaled)
pd.DataFrame({
    "pos": x_axis,
    "weighted_density_gene": wd_gene,
    "weighted_density_intergenic": wd_neg
}).to_csv(wd_tsv, sep="\t", index=False)
print(f"Wrote: {wd_tsv}")

# display scale 0–1 ACROSS the two curves (don't rescale per-curve)
gmax = max(wd_gene.max(), wd_neg.max(), 1e-12)
plt.figure(figsize=(12, 5))
plt.plot(x_axis, wd_gene / gmax, label="genes", lw=2)
plt.plot(x_axis, wd_neg  / gmax, label="intergenic", lw=2)
plt.axvspan(-UP, 0, color="green", alpha=0.15, label="TSS −1000 bp")
plt.axvspan(BINS_BODY, BINS_BODY + DOWN, color="red", alpha=0.15, label="TTS +1000 bp")
plt.xticks([-UP, 0, BINS_BODY, BINS_BODY + DOWN], ["-1000", "TSS", "TTS", "+1000"])
plt.xlabel("Position (bp / normalized body)")
plt.ylabel("Weighted density (0–1 across groups)")
plt.title("GRO-seq weighted density across genes (shape × magnitude)")
plt.ylim(0, 1)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(plot_wd_png, dpi=300)
plt.show()
print(f"Wrote: {plot_wd_png}")


In [ ]:
#!/usr/bin/env python3
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ======================
# Inputs
# ======================
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/predicted_gene_boundaries.bed"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"

# GRO-seq normalized readcounts file(s)
# Accepts either:
#   (A) header with columns incl. 'chr' and ('pos' or 'site'), plus a value column
#   (B) 3 cols (no header): chr pos value
GRO_DIR  = "/rhome/zli529/lab/SRA_toolkit/RNAseq_7stages/processed_data/LT"
GRO_GLOB = "LT_coverage.txt"  # wildcard OK

# Outputs
out_dir        = GRO_DIR
plot_mean_png  = os.path.join(out_dir, "GROseq_genes_vs_intergenic_mean.png")
plot_wd_png    = os.path.join(out_dir, "GROseq_genes_vs_intergenic_weighted_density.png")
profiles_tsv   = os.path.join(out_dir, "GROseq_genes_vs_intergenic_profiles.tsv")
wd_tsv         = os.path.join(out_dir, "GROseq_genes_vs_intergenic_weighted_density.tsv")

# ======================
# Parameters
# ======================
UP = 1000          # bp upstream of TSS
DOWN = 1000        # bp downstream of TTS
BINS_BODY = 1000   # bins for normalized gene body
SMOOTH_SD_BP = 25  # Gaussian smoothing SD (bp) for densities; set 0 to disable
CHUNK_SIZE = 2_000_000  # lines per chunk when reading GRO-seq

# ======================
# Helpers
# ======================
def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

def to_prob_density(y):
    y = np.asarray(y, dtype=float)
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def detect_header(path):
    with open(path, "r") as f:
        for line in f:
            if not line.strip() or line.lstrip().startswith("#"):
                continue
            toks = line.strip().split()
            low = [t.lower() for t in toks]
            # consider header if 'chr' is present and either 'pos' or 'site'
            return ("chr" in low) and (("pos" in low) or ("site" in low))
    return False

def load_bed_flexible(path):
    """
    Robust BED loader:
      - Supports >=3 columns with unknown extras
      - Detects start/end numeric columns after 'chr'
      - Detects strand column if present (else '+')
      - Converts to 1-based coordinates internally
    Returns DataFrame with columns: chr, start1, end1, strand
    """
    # Read raw (no header). Drop track/browser/comment lines.
    raw = pd.read_csv(path, sep=r"\s+|\t|,", engine="python", header=None, comment="#", dtype=str)
    if raw.empty:
        raise ValueError(f"BED empty: {path}")

    # Drop 'track'/'browser' rows if any
    m_bad = raw.iloc[:,0].astype(str).str.lower().isin(["track","browser"])
    if m_bad.any():
        raw = raw[~m_bad]
        if raw.empty:
            raise ValueError("BED has only 'track/browser' lines.")

    # Identify chr column (assume first)
    chr_col = 0

    # Find numeric candidates for start/end among subsequent columns
    num_mask = {}
    for c in range(1, raw.shape[1]):
        s = pd.to_numeric(raw.iloc[:, c], errors="coerce")
        frac_num = s.notna().mean()
        num_mask[c] = (frac_num > 0.95)  # mostly numeric
    num_cols = [c for c, ok in num_mask.items() if ok]
    if len(num_cols) < 2:
        raise ValueError("Could not find two numeric columns for start/end in BED.")

    # Choose the first two numeric columns as start0 and end0 (BED 0-based half-open)
    start_col0, end_col0 = num_cols[0], num_cols[1]

    start0 = pd.to_numeric(raw.iloc[:, start_col0], errors="coerce")
    end0   = pd.to_numeric(raw.iloc[:, end_col0], errors="coerce")
    # If needed, swap so start0 <= end0
    swap_mask = start0 > end0
    if swap_mask.any():
        tmp = start0.copy()
        start0[swap_mask] = end0[swap_mask]
        end0[swap_mask]   = tmp[swap_mask]

    # Detect strand column: prefer a column with only '+'/'-'
    strand = None
    for c in range(1, raw.shape[1]):
        vals = raw.iloc[:, c].astype(str).str.strip()
        is_pm = vals.isin(["+","-"]).mean()
        if is_pm > 0.95:
            strand = vals
            break
    if strand is None:
        strand = pd.Series("+", index=raw.index)  # assume +

    # Build clean DataFrame (convert 0-based -> 1-based for internal bp indexing)
    # BED: chromStart is 0-based inclusive; chromEnd is 0-based exclusive.
    start1 = (start0 + 1).astype("Int64")
    end1   = end0.astype("Int64")

    df = pd.DataFrame({
        "chr": raw.iloc[:, chr_col].astype(str),
        "start1": start1,
        "end1": end1,
        "strand": strand.astype(str)
    }).dropna()

    # enforce integer dtype
    df["start1"] = df["start1"].astype(np.int64)
    df["end1"]   = df["end1"].astype(np.int64)
    return df

# ======================
# Load genes / negatives
# ======================
# Load predicted gene boundaries from flexible BED
genes_bed = load_bed_flexible(mRNA_file)

# Build mRNA DataFrame compatible with downstream code: chr/start/end/strand (1-based)
mRNA = pd.DataFrame({
    "chr":    genes_bed["chr"].astype("string"),
    "start":  genes_bed["start1"].astype(np.int64),
    "end":    genes_bed["end1"].astype(np.int64),
    "gene_id": pd.RangeIndex(0, len(genes_bed)).astype(str),  # dummy IDs if needed
    "strand": genes_bed["strand"].astype("string"),
})
# TSS/TTS in 1-based space
mRNA["tss"] = np.where(mRNA["strand"] == "+", mRNA["start"], mRNA["end"])
mRNA["tts"] = np.where(mRNA["strand"] == "+", mRNA["end"],   mRNA["start"])

# negatives: chr start end id strand  (center window)
neg = pd.read_csv(
    neg_file, sep=r"\s+|,|\t", header=None,
    names=["chr","start","end","id","strand"],
    dtype={"chr":"string","start":"int64","end":"int64","id":"string","strand":"string"},
    engine="python"
)
neg["center"] = ((neg["start"] + neg["end"]) // 2).astype("int64")

# ======================
# Build GRO-seq coverage dict (chr -> Series(pos -> value))
# ======================
files = sorted(glob.glob(os.path.join(GRO_DIR, GRO_GLOB)))
assert files, f"No files matched {os.path.join(GRO_DIR, GRO_GLOB)}"

cov_by_chr = {}

for path in files:
    has_hdr = detect_header(path)
    if has_hdr:
        # Read header then resolve column names flexibly
        head_df = pd.read_csv(path, sep=r"\s+|\t|,", engine="python", nrows=5)
        cols = {c.lower(): c for c in head_df.columns}
        chr_c  = cols.get("chr") or list(head_df.columns)[0]
        pos_c  = cols.get("pos") or cols.get("site")  # accept 'site' too
        val_c  = cols.get("normalized_count") or cols.get("value") or cols.get("count") or list(head_df.columns)[-1]
        usecols = [chr_c, pos_c, val_c]

        for chunk in pd.read_csv(path, sep=r"\s+|\t|,", engine="python", usecols=usecols,
                                 chunksize=CHUNK_SIZE, dtype={chr_c:"string"}):
            chunk = chunk.rename(columns={chr_c:"chr", pos_c:"pos", val_c:"val"})
            # coerce numerics
            chunk["pos"] = pd.to_numeric(chunk["pos"], errors="coerce")
            chunk["val"] = pd.to_numeric(chunk["val"], errors="coerce")
            chunk = chunk.dropna()
            if chunk.empty: 
                continue
            chunk["pos"] = chunk["pos"].astype(np.int64, copy=False)
            grp = chunk.groupby(["chr","pos"], as_index=False, sort=False)["val"].sum()
            for chrom, sub in grp.groupby("chr", sort=False):
                s_new = pd.Series(sub["val"].to_numpy(), index=sub["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new
    else:
        # No header: 3 columns (chr pos value). Third may have ';...' trailing.
        for chunk in pd.read_csv(path, sep=r"\s+", header=None, names=["chr","pos","raw"],
                                 usecols=[0,1,2], engine="c", chunksize=CHUNK_SIZE,
                                 dtype={"chr":"string","pos":"Int64","raw":"string"},
                                 na_filter=False, on_bad_lines="skip"):
            raw = chunk["raw"].str.replace(r";.*$", "", regex=True)
            sub = pd.DataFrame({
                "chr": chunk["chr"].astype("string"),
                "pos": pd.to_numeric(chunk["pos"], errors="coerce"),
                "val": pd.to_numeric(raw, errors="coerce")
            }).dropna()
            if sub.empty: 
                continue
            sub["pos"] = sub["pos"].astype(np.int64, copy=False)
            grp = sub.groupby(["chr","pos"], as_index=False, sort=False)["val"].sum()
            for chrom, g in grp.groupby("chr", sort=False):
                s_new = pd.Series(g["val"].to_numpy(), index=g["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new

# tidy
for k, s in list(cov_by_chr.items()):
    s.index = s.index.astype(np.int64, copy=False)
    cov_by_chr[k] = s.sort_index()

# ======================
# Meta-gene profile builders
# ======================
def gene_profile(row, cov_by_chr, up=UP, down=DOWN, bins_body=BINS_BODY):
    chrom = row["chr"]
    s = cov_by_chr.get(chrom)
    if s is None or s.empty:
        return None

    start, end, strand = int(row["start"]), int(row["end"]), row["strand"]
    tss = start if strand == "+" else end
    tts = end   if strand == "+" else start

    region_start = tss - up if strand == "+" else tts - up
    region_end   = tts + down if strand == "+" else tss + down  # exclusive
    length = region_end - region_start
    if length <= 0:
        return None

    idx  = np.arange(region_start, region_end, dtype=np.int64)
    vals = s.reindex(idx, fill_value=0.0).to_numpy()

    if strand == "-":
        vals = vals[::-1]

    prom = vals[:up]
    body = vals[up: length - down]
    downv = vals[length - down:]

    if body.size > 1:
        body_x = np.linspace(0, body.size - 1, bins_body)
        body_norm = np.interp(body_x, np.arange(body.size), body)
    else:
        body_norm = np.zeros(bins_body, dtype=float)

    return np.concatenate([prom, body_norm, downv])

def neg_profile(row, cov_by_chr, up=UP, down=DOWN, bins_body=BINS_BODY):
    chrom = row["chr"]
    s = cov_by_chr.get(chrom)
    if s is None or s.empty:
        return None
    center = int(row["center"])
    region_start = center - up
    region_end   = center + bins_body + down
    idx = np.arange(region_start, region_end, dtype=np.int64)
    vals = s.reindex(idx, fill_value=0.0).to_numpy()
    return vals

def average_profiles(df, cov_by_chr, fn):
    profs = []
    for _, row in df.iterrows():
        p = fn(row, cov_by_chr)
        if p is not None:
            profs.append(p)
    if not profs:
        return np.zeros(UP + BINS_BODY + DOWN, dtype=float), 0
    arr = np.vstack(profs)
    return arr.mean(axis=0), arr.shape[0]

# ======================
# Compute mean profiles
# ======================
mean_gene, n_gene = average_profiles(mRNA, cov_by_chr, gene_profile)
mean_neg,  n_neg  = average_profiles(neg,  cov_by_chr, neg_profile)

# Save raw mean profiles
x_axis = np.arange(-UP, BINS_BODY + DOWN, dtype=int)
pd.DataFrame({
    "pos": x_axis,
    "mean_GRO_gene": mean_gene,
    "mean_GRO_intergenic": mean_neg,
    "n_gene_windows": np.full_like(x_axis, n_gene),
    "n_intergenic_windows": np.full_like(x_axis, n_neg),
}).to_csv(profiles_tsv, sep="\t", index=False)
print(f"Wrote: {profiles_tsv}")

# ======================
# Plot mean (absolute signal)
# ======================
plt.figure(figsize=(12, 5))
plt.plot(x_axis, mean_gene, label=f"genes (n={n_gene})", lw=2)
plt.plot(x_axis, mean_neg,  label=f"intergenic (n={n_neg})", lw=2)
plt.axvspan(-UP, 0, color="green", alpha=0.15, label="TSS −1000 bp")
plt.axvspan(BINS_BODY, BINS_BODY + DOWN, color="red", alpha=0.15, label="TTS +1000 bp")
plt.xticks([-UP, 0, BINS_BODY, BINS_BODY + DOWN], ["-1000", "TSS", "TTS", "+1000"])
plt.xlabel("Position (bp / normalized body)")
plt.ylabel("Mean GRO-seq (normalized)")
plt.title("GRO-seq coverage across genes (absolute mean)")
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(plot_mean_png, dpi=300)
plt.show()
print(f"Wrote: {plot_mean_png}")

# ======================
# Weighted density (shape × magnitude)
# ======================
dens_gene = to_prob_density(mean_gene)
dens_neg  = to_prob_density(mean_neg)

if SMOOTH_SD_BP > 0:
    dens_gene = to_prob_density(gaussian_smooth(dens_gene, sd_bp=SMOOTH_SD_BP))
    dens_neg  = to_prob_density(gaussian_smooth(dens_neg,  sd_bp=SMOOTH_SD_BP))

mass_gene = float(np.nansum(mean_gene))
mass_neg  = float(np.nansum(mean_neg))

max_mass = max(mass_gene, mass_neg, 1e-12)
wd_gene = dens_gene * (mass_gene / max_mass)
wd_neg  = dens_neg  * (mass_neg  / max_mass)

pd.DataFrame({
    "pos": x_axis,
    "weighted_density_gene": wd_gene,
    "weighted_density_intergenic": wd_neg
}).to_csv(wd_tsv, sep="\t", index=False)
print(f"Wrote: {wd_tsv}")

gmax = max(wd_gene.max(), wd_neg.max(), 1e-12)
plt.figure(figsize=(12, 5))
plt.plot(x_axis, wd_gene / gmax, label="genes", lw=2)
plt.plot(x_axis, wd_neg  / gmax, label="intergenic", lw=2)
plt.axvspan(-UP, 0, color="green", alpha=0.15, label="TSS −1000 bp")
plt.axvspan(BINS_BODY, BINS_BODY + DOWN, color="red", alpha=0.15, label="TTS +1000 bp")
plt.xticks([-UP, 0, BINS_BODY, BINS_BODY + DOWN], ["-1000", "TSS", "TTS", "+1000"])
plt.xlabel("Position (bp / normalized body)")
plt.ylabel("Weighted density (0–1 across groups)")
plt.title("GRO-seq weighted density across genes (shape × magnitude)")
plt.ylim(0, 1)
plt.legend(frameon=False)
plt.tight_layout()
plt.savefig(plot_wd_png, dpi=300)
plt.show()
print(f"Wrote: {plot_wd_png}")


In [ ]:
#!/usr/bin/env python3
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------
# Inputs
# ------------------------
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/reference_gene_boundaries.csv"
# neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"
# e.g., to use your telomere negatives instead:
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/random_TSS_from_telomeres.tsv"

# GRO-seq file or glob (can be one file or many)
POLYA_DIR  = "/rhome/zli529/lab/SRA_toolkit/RNAseq_7stages/processed_data/LT"
POLYA_GLOB = "LT_coverage.txt"  # supports wildcards

# Outputs
out_dir       = POLYA_DIR
plot_mean_png = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_genes_vs_intergenic.png")
plot_wd_png   = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_weighted_density.png")
profiles_tsv  = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_profiles.tsv")
wd_tsv        = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_weighted_density.tsv")

# Window
UP = 500
DOWN = 500
OFFSETS = np.arange(-UP, DOWN + 1, dtype=np.int64)   # -500..+500 (1001 points)
SMOOTH_SD_BP = 25  # set 0 to disable smoothing for densities

# ------------------------
# Helpers
# ------------------------
def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

def to_prob_density(y):
    y = np.asarray(y, dtype=float)
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def detect_header(path):
    # Detects 'chr/pos/count/normalized_count' anywhere in first non-empty, non-comment line
    with open(path, "r") as f:
        for line in f:
            if not line.strip():
                continue
            if line.lstrip().startswith("#"):
                continue
            tokens = line.strip().split()
            return any(t.lower() in {"chr", "pos", "count", "normalized_count"} for t in tokens)
    return False

def has_header_three_cols(path):
    # For negatives (3-col form), detect header with 'chr' and 'position'
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            toks = s.split()
            if len(toks) >= 3 and toks[0].lower() == "chr" and toks[1].lower().startswith("pos"):
                return True
            return False
    return False

# ------------------------
# Load TSS/TTS (reference genes)
# Expect CSV/TSV with: chr,start,end,gene_id,strand
# ------------------------
mRNA = pd.read_csv(
    mRNA_file, sep=r"\s+|,|\t", header=None,
    names=["chr","start","end","gene_id","strand"],
    dtype={"chr":"string","start":"int64","end":"int64","gene_id":"string","strand":"string"},
    engine="python"
)
mRNA["tss"] = np.where(mRNA["strand"] == "+", mRNA["start"], mRNA["end"])
mRNA["tts"] = np.where(mRNA["strand"] == "+", mRNA["end"],   mRNA["start"])
tss_by_chr = {c: g["tss"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}
tts_by_chr = {c: g["tts"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}

# ------------------------
# Load negatives (auto-detect schema)
# Supports:
#  (A) 3-col with header:  chr  position  strand        -> center = position
#  (B) 5-col no-header:    chr  col2  col3  id  strand  -> center = col2 if '+' else col3
#  (C) 3-col no-header:    chr  pos   strand            -> center = pos
# ------------------------
with open(neg_file, "r") as f:
    first_data = None
    for line in f:
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        first_data = s.split()
        break
num_cols = len(first_data) if first_data else 0

if num_cols >= 5:
    # (B) 5 columns (assume no header): chr col2 col3 id strand
    neg_df = pd.read_csv(
        neg_file, sep=r"\s+|,|\t", header=None,
        names=["chr","col2","col3","id","strand"],
        dtype={"chr":"string","col2":"int64","col3":"int64","id":"string","strand":"string"},
        engine="python"
    )
    neg_df["center"] = np.where(neg_df["strand"] == "+", neg_df["col2"], neg_df["col3"])

elif num_cols == 3:
    if has_header_three_cols(neg_file):
        # (A) 3 columns with header: chr position strand
        neg_df = pd.read_csv(
            neg_file, sep=r"\s+|,|\t", header=0,
            usecols=[0,1,2],
            dtype={"chr":"string","position":"int64","strand":"string"},
            engine="python"
        )
        neg_df = neg_df.rename(columns={"position":"pos"})
        neg_df["center"] = neg_df["pos"].astype("int64")
    else:
        # (C) 3 columns no header: chr pos strand
        neg_df = pd.read_csv(
            neg_file, sep=r"\s+|,|\t", header=None,
            names=["chr","pos","strand"],
            dtype={"chr":"string","pos":"int64","strand":"string"},
            engine="python"
        )
        neg_df["center"] = neg_df["pos"].astype("int64")
else:
    raise ValueError(f"Unrecognized negatives format in {neg_file}: saw {num_cols} columns.")

neg_by_chr = {c: g["center"].to_numpy(dtype=np.int64) for c,g in neg_df.groupby("chr")}

# ------------------------
# Stream GRO-seq → chr->Series(pos->value) (summing across files)
# Robust to header/no-header and stray text after ';'
# ------------------------
files = sorted(glob.glob(os.path.join(POLYA_DIR, POLYA_GLOB)))
assert files, f"No files matched {os.path.join(POLYA_DIR, POLYA_GLOB)}"

cov_by_chr = {}  # chr -> Series(index=pos, values=normalized_count sum)

for f in files:
    has_hdr = detect_header(f)
    if has_hdr:
        # Expect columns: chr pos count normalized_count (we take 0,1,3)
        usecols = [0, 1, 3]
        for chunk in pd.read_csv(
            f,
            sep=r"\s+",
            header=0,
            usecols=usecols,
            engine="c",
            chunksize=2_000_000,
            dtype={0: "string", 1: "int64", 3: "float64"},
            on_bad_lines="skip"
        ):
            chunk.columns = ["chr", "pos", "normalized_count"]
            sub = chunk.dropna()
            if sub.empty:
                continue
            sub["pos"] = sub["pos"].astype(np.int64, copy=False)
            grp = sub.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
            for chrom, g in grp.groupby("chr", sort=False):
                s_new = pd.Series(g["normalized_count"].to_numpy(), index=g["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new
    else:
        # No header: assume 3 cols: chr pos value (value may contain ';...' trailing)
        for chunk in pd.read_csv(
            f,
            sep=r"\s+",
            header=None,
            names=["chr","pos","raw"],
            usecols=[0,1,2],
            engine="c",
            chunksize=2_000_000,
            dtype={"chr":"string","pos":"Int64","raw":"string"},
            na_filter=False,
            on_bad_lines="skip"
        ):
            raw_clean = chunk["raw"].str.replace(r";.*$", "", regex=True)
            sub = pd.DataFrame({
                "chr": chunk["chr"].astype("string"),
                "pos": pd.to_numeric(chunk["pos"], errors="coerce"),
                "normalized_count": pd.to_numeric(raw_clean, errors="coerce")
            }).dropna()
            if sub.empty:
                continue
            sub["pos"] = sub["pos"].astype(np.int64, copy=False)
            grp = sub.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
            for chrom, g in grp.groupby("chr", sort=False):
                s_new = pd.Series(g["normalized_count"].to_numpy(), index=g["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new

# tidy: int index & sort
for k, s in list(cov_by_chr.items()):
    s.index = s.index.astype(np.int64, copy=False)
    cov_by_chr[k] = s.sort_index()

# ------------------------
# Mean profiles (fill missing signal with 0 so every site contributes everywhere)
# ------------------------
def mean_profile_from_positions(pos_by_chr, cov_by_chr, offsets=OFFSETS):
    total = np.zeros(offsets.size, dtype=np.float64)
    count = 0
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0:
            continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty:
            continue
        pos_mat = positions[:, None] + offsets[None, :]           # n_pos x len(offsets)
        pulled = series.reindex(pos_mat.ravel(), fill_value=0.0).to_numpy().reshape(pos_mat.shape)
        total += pulled.sum(axis=0)
        count += positions.size
    mean = (total / count) if count > 0 else total
    return mean, np.full(offsets.size, count, dtype=np.int64)

# TSS
mean_tss_genes, n_tss_genes = mean_profile_from_positions(tss_by_chr, cov_by_chr, OFFSETS)
mean_tss_negs,  n_tss_negs  = mean_profile_from_positions(neg_by_chr,  cov_by_chr, OFFSETS)

# TTS
mean_tts_genes, n_tts_genes = mean_profile_from_positions(tts_by_chr, cov_by_chr, OFFSETS)
mean_tts_negs,  n_tts_negs  = mean_profile_from_positions(neg_by_chr,  cov_by_chr, OFFSETS)

# ------------------------
# Save TSV of mean profiles
# ------------------------
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_GRO_TSS_genes": mean_tss_genes,
    "mean_GRO_TSS_intergenic": mean_tss_negs,
    "n_TSS_gene_windows": n_tss_genes,
    "n_TSS_intergenic_windows": n_tss_negs,
    "mean_GRO_TTS_genes": mean_tts_genes,
    "mean_GRO_TTS_intergenic": mean_tts_negs,
    "n_TTS_gene_windows": n_tts_genes,
    "n_TTS_intergenic_windows": n_tts_negs,
})
profiles.to_csv(profiles_tsv, sep="\t", index=False)
print(f"Wrote: {profiles_tsv}")

# ------------------------
# Plot mean CPM-style profiles (absolute signal)
# ------------------------
plt.figure(figsize=(11, 4.5))
plt.subplot(1, 2, 1)
plt.plot(OFFSETS, mean_tss_genes, label="genes", linewidth=2)
plt.plot(OFFSETS, mean_tss_negs,  label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("GRO-seq around TSS (±500 bp)")
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Mean GRO-seq (normalized)")
plt.legend(frameon=False)

plt.subplot(1, 2, 2)
plt.plot(OFFSETS, mean_tts_genes, label="genes", linewidth=2)
plt.plot(OFFSETS, mean_tts_negs,  label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("GRO-seq around TTS (±500 bp)")  # fixed to match window
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean GRO-seq (normalized)")
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig(plot_mean_png, dpi=300)
plt.show()
print(f"Wrote: {plot_mean_png}")

# ------------------------
# Weighted density (shape × magnitude), with optional smoothing
# ------------------------
# Build probability densities
dens_tss_genes = to_prob_density(mean_tss_genes)
dens_tss_negs  = to_prob_density(mean_tss_negs)
dens_tts_genes = to_prob_density(mean_tts_genes)
dens_tts_negs  = to_prob_density(mean_tts_negs)

if SMOOTH_SD_BP > 0:
    dens_tss_genes = to_prob_density(gaussian_smooth(dens_tss_genes, sd_bp=SMOOTH_SD_BP))
    dens_tss_negs  = to_prob_density(gaussian_smooth(dens_tss_negs,  sd_bp=SMOOTH_SD_BP))
    dens_tts_genes = to_prob_density(gaussian_smooth(dens_tts_genes, sd_bp=SMOOTH_SD_BP))
    dens_tts_negs  = to_prob_density(gaussian_smooth(dens_tts_negs,  sd_bp=SMOOTH_SD_BP))

# Panel-wise masses (absolute signal)
mass_tss_genes = float(np.nansum(mean_tss_genes))
mass_tss_negs  = float(np.nansum(mean_tss_negs))
mass_tts_genes = float(np.nansum(mean_tts_genes))
mass_tts_negs  = float(np.nansum(mean_tts_negs))

# Apply weights (normalize weights within panel so largest mass == 1)
def weight_panel(d1, d2, m1, m2):
    mm = max(m1, m2, 1e-12)
    return d1 * (m1 / mm), d2 * (m2 / mm)

wd_tss_genes, wd_tss_negs = weight_panel(dens_tss_genes, dens_tss_negs, mass_tss_genes, mass_tss_negs)
wd_tts_genes, wd_tts_negs = weight_panel(dens_tts_genes, dens_tts_negs, mass_tts_genes, mass_tts_negs)

# Save weighted densities (raw, before display scaling)
wd_df = pd.DataFrame({
    "offset_bp": OFFSETS,
    "weighted_density_TSS_genes": wd_tss_genes,
    "weighted_density_TSS_intergenic": wd_tss_negs,
    "weighted_density_TTS_genes": wd_tts_genes,
    "weighted_density_TTS_intergenic": wd_tts_negs,
})
wd_df.to_csv(wd_tsv, sep="\t", index=False)
print(f"Wrote: {wd_tsv}")

# Display scaling (0–1 per panel across the two curves)
gmax_tss = max(wd_tss_genes.max(), wd_tss_negs.max(), 1e-12)
gmax_tts = max(wd_tts_genes.max(), wd_tts_negs.max(), 1e-12)

plt.figure(figsize=(11, 4.5))

plt.subplot(1, 2, 1)
plt.plot(OFFSETS, wd_tss_genes / gmax_tss, label="genes", linewidth=2)
plt.plot(OFFSETS, wd_tss_negs  / gmax_tss, label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Weighted density around TSS (±500 bp)")
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Weighted density (0–1 across panel)")
plt.ylim(0, 1)
plt.legend(frameon=False)

plt.subplot(1, 2, 2)
plt.plot(OFFSETS, wd_tts_genes / gmax_tts, label="genes", linewidth=2)
plt.plot(OFFSETS, wd_tts_negs  / gmax_tts, label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Weighted density around TTS (±500 bp)")
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Weighted density (0–1 across panel)")
plt.ylim(0, 1)
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig(plot_wd_png, dpi=300)
plt.show()
print(f"Wrote: {plot_wd_png}")


In [ ]:
#!/usr/bin/env python3
import os, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------
# Inputs
# ------------------------
# Use your BED6 file here:
mRNA_file = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/predicted_gene_boundaries.bed"

# Negatives (works with 3-col header 'chr position strand', 3-col no-header, or legacy 5-col)
# neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/training_features_values/negatives_2k.tsv"
neg_file  = "/rhome/zli529/lab/LncRNA_chip_prediction/Final/window1000_bin100/New_genome_features/random_TSS_from_telomeres.tsv"

# GRO-seq file or glob (can be one file or many)
POLYA_DIR  = "/rhome/zli529/lab/SRA_toolkit/RNAseq_7stages/processed_data/LT"
POLYA_GLOB = "LT_coverage.txt"  # supports wildcards

# Outputs
out_dir       = POLYA_DIR
plot_mean_png = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_genes_vs_intergenic.png")
plot_wd_png   = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_weighted_density.png")
profiles_tsv  = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_profiles.tsv")
wd_tsv        = os.path.join(out_dir, "2201reference_TSS_TTS_GRO-seq_weighted_density.tsv")

# Window
UP = 500
DOWN = 500
OFFSETS = np.arange(-UP, DOWN + 1, dtype=np.int64)   # -500..+500 (1001 points)
SMOOTH_SD_BP = 25  # set 0 to disable smoothing for densities

# ------------------------
# Helpers
# ------------------------
def gaussian_smooth(y, sd_bp=25.0, step_bp=1.0):
    y = np.asarray(y, dtype=float)
    if sd_bp is None or sd_bp <= 0:
        return y
    sd_steps = sd_bp / step_bp
    radius = max(1, int(4 * sd_steps))
    x = np.arange(-radius, radius + 1, dtype=float)
    ker = np.exp(-0.5 * (x / sd_steps) ** 2)
    ker /= ker.sum()
    return np.convolve(y, ker, mode="same")

def to_prob_density(y):
    y = np.asarray(y, dtype=float)
    y[y < 0] = 0.0
    s = y.sum()
    return y / s if s > 0 else y

def detect_header(path):
    # Detects 'chr/pos/count/normalized_count' anywhere in first non-empty, non-comment line (for GRO files)
    with open(path, "r") as f:
        for line in f:
            if not line.strip():
                continue
            if line.lstrip().startswith("#"):
                continue
            tokens = line.strip().split()
            return any(t.lower() in {"chr", "pos", "count", "normalized_count"} for t in tokens)
    return False

def has_header_three_cols(path):
    # For negatives (3-col form), detect header with 'chr' and 'position'
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            toks = s.split()
            if len(toks) >= 3 and toks[0].lower() == "chr" and toks[1].lower().startswith("pos"):
                return True
            return False
    return False

def sniff_first_data_tokens(path):
    with open(path, "r") as f:
        for line in f:
            s = line.strip()
            if not s or s.startswith("#"):
                continue
            return s.split()
    return None

# ------------------------
# Load TSS/TTS (reference genes) - supports BED6 or CSV/TSV with chr,start,end,gene_id,strand
# ------------------------
first = sniff_first_data_tokens(mRNA_file)
if first is None:
    raise ValueError(f"No data found in {mRNA_file}")

# BED6 if >=6 tokens and 6th is a strand symbol
is_bed6_like = (len(first) >= 6 and first[5] in {"+", "-", ".", "?"})

if is_bed6_like:
    # BED6: chrom, start, end, name, score, strand
    mRNA = pd.read_csv(
        mRNA_file, sep=r"\s+|,|\t", header=None,
        names=["chr","start","end","name","score","strand"],
        dtype={"chr":"string","start":"int64","end":"int64","name":"string","score":"Int64","strand":"string"},
        engine="python"
    )
    # Map to expected columns
    mRNA["gene_id"] = mRNA["name"]
    mRNA = mRNA[["chr","start","end","gene_id","strand"]]
else:
    # Expect CSV/TSV: chr,start,end,gene_id,strand (headerless)
    mRNA = pd.read_csv(
        mRNA_file, sep=r"\s+|,|\t", header=None,
        names=["chr","start","end","gene_id","strand"],
        dtype={"chr":"string","start":"int64","end":"int64","gene_id":"string","strand":"string"},
        engine="python"
    )

# Optional: drop duplicates (the BED appears to have repeated rows)
mRNA = mRNA.drop_duplicates(ignore_index=True)

# Compute TSS/TTS
mRNA["tss"] = np.where(mRNA["strand"] == "+", mRNA["start"], mRNA["end"])
mRNA["tts"] = np.where(mRNA["strand"] == "+", mRNA["end"],   mRNA["start"])
tss_by_chr = {c: g["tss"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}
tts_by_chr = {c: g["tts"].to_numpy(dtype=np.int64) for c,g in mRNA.groupby("chr")}

# ------------------------
# Load negatives (auto-detect schema)
# Supports:
#  (A) 3-col with header:  chr  position  strand        -> center = position
#  (B) 5-col no-header:    chr  col2  col3  id  strand  -> center = col2 if '+' else col3
#  (C) 3-col no-header:    chr  pos   strand            -> center = pos
# ------------------------
first_neg = sniff_first_data_tokens(neg_file)
num_cols = len(first_neg) if first_neg else 0

if num_cols >= 5:
    # (B) 5 columns (assume no header): chr col2 col3 id strand
    neg_df = pd.read_csv(
        neg_file, sep=r"\s+|,|\t", header=None,
        names=["chr","col2","col3","id","strand"],
        dtype={"chr":"string","col2":"int64","col3":"int64","id":"string","strand":"string"},
        engine="python"
    )
    neg_df["center"] = np.where(neg_df["strand"] == "+", neg_df["col2"], neg_df["col3"])

elif num_cols == 3:
    if has_header_three_cols(neg_file):
        # (A) 3 columns with header: chr position strand
        neg_df = pd.read_csv(
            neg_file, sep=r"\s+|,|\t", header=0,
            usecols=[0,1,2],
            dtype={"chr":"string","position":"int64","strand":"string"},
            engine="python"
        )
        neg_df = neg_df.rename(columns={"position":"pos"})
        neg_df["center"] = neg_df["pos"].astype("int64")
    else:
        # (C) 3 columns no header: chr pos strand
        neg_df = pd.read_csv(
            neg_file, sep=r"\s+|,|\t", header=None,
            names=["chr","pos","strand"],
            dtype={"chr":"string","pos":"int64","strand":"string"},
            engine="python"
        )
        neg_df["center"] = neg_df["pos"].astype("int64")
else:
    raise ValueError(f"Unrecognized negatives format in {neg_file}: saw {num_cols} columns.")

neg_by_chr = {c: g["center"].to_numpy(dtype=np.int64) for c,g in neg_df.groupby("chr")}

# ------------------------
# Stream GRO-seq → chr->Series(pos->value) (summing across files)
# Robust to header/no-header and stray text after ';'
# ------------------------
files = sorted(glob.glob(os.path.join(POLYA_DIR, POLYA_GLOB)))
assert files, f"No files matched {os.path.join(POLYA_DIR, POLYA_GLOB)}"

cov_by_chr = {}  # chr -> Series(index=pos, values=normalized_count sum)

for f in files:
    has_hdr = detect_header(f)
    if has_hdr:
        # Expect columns: chr pos count normalized_count (we take 0,1,3)
        usecols = [0, 1, 3]
        for chunk in pd.read_csv(
            f,
            sep=r"\s+",
            header=0,
            usecols=usecols,
            engine="c",
            chunksize=2_000_000,
            dtype={0: "string", 1: "int64", 3: "float64"},
            on_bad_lines="skip"
        ):
            chunk.columns = ["chr", "pos", "normalized_count"]
            sub = chunk.dropna()
            if sub.empty:
                continue
            sub["pos"] = sub["pos"].astype(np.int64, copy=False)
            grp = sub.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
            for chrom, g in grp.groupby("chr", sort=False):
                s_new = pd.Series(g["normalized_count"].to_numpy(), index=g["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new
    else:
        # No header: assume 3 cols: chr pos value (value may contain ';...' trailing)
        for chunk in pd.read_csv(
            f,
            sep=r"\s+",
            header=None,
            names=["chr","pos","raw"],
            usecols=[0,1,2],
            engine="c",
            chunksize=2_000_000,
            dtype={"chr":"string","pos":"Int64","raw":"string"},
            na_filter=False,
            on_bad_lines="skip"
        ):
            raw_clean = chunk["raw"].str.replace(r";.*$", "", regex=True)
            sub = pd.DataFrame({
                "chr": chunk["chr"].astype("string"),
                "pos": pd.to_numeric(chunk["pos"], errors="coerce"),
                "normalized_count": pd.to_numeric(raw_clean, errors="coerce")
            }).dropna()
            if sub.empty:
                continue
            sub["pos"] = sub["pos"].astype(np.int64, copy=False)
            grp = sub.groupby(["chr","pos"], as_index=False, sort=False)["normalized_count"].sum()
            for chrom, g in grp.groupby("chr", sort=False):
                s_new = pd.Series(g["normalized_count"].to_numpy(), index=g["pos"].to_numpy())
                key = str(chrom)
                cov_by_chr[key] = cov_by_chr[key].add(s_new, fill_value=0.0) if key in cov_by_chr else s_new

# tidy: int index & sort
for k, s in list(cov_by_chr.items()):
    s.index = s.index.astype(np.int64, copy=False)
    cov_by_chr[k] = s.sort_index()

# ------------------------
# Mean profiles (fill missing signal with 0 so every site contributes everywhere)
# ------------------------
def mean_profile_from_positions(pos_by_chr, cov_by_chr, offsets=OFFSETS):
    total = np.zeros(offsets.size, dtype=np.float64)
    count = 0
    for chrom, positions in pos_by_chr.items():
        if positions.size == 0:
            continue
        series = cov_by_chr.get(chrom)
        if series is None or series.empty:
            continue
        pos_mat = positions[:, None] + offsets[None, :]           # n_pos x len(offsets)
        pulled = series.reindex(pos_mat.ravel(), fill_value=0.0).to_numpy().reshape(pos_mat.shape)
        total += pulled.sum(axis=0)
        count += positions.size
    mean = (total / count) if count > 0 else total
    return mean, np.full(offsets.size, count, dtype=np.int64)

# TSS
mean_tss_genes, n_tss_genes = mean_profile_from_positions(tss_by_chr, cov_by_chr, OFFSETS)
mean_tss_negs,  n_tss_negs  = mean_profile_from_positions(neg_by_chr,  cov_by_chr, OFFSETS)

# TTS
mean_tts_genes, n_tts_genes = mean_profile_from_positions(tts_by_chr, cov_by_chr, OFFSETS)
mean_tts_negs,  n_tts_negs  = mean_profile_from_positions(neg_by_chr,  cov_by_chr, OFFSETS)

# ------------------------
# Save TSV of mean profiles
# ------------------------
profiles = pd.DataFrame({
    "offset_bp": OFFSETS,
    "mean_GRO_TSS_genes": mean_tss_genes,
    "mean_GRO_TSS_intergenic": mean_tss_negs,
    "n_TSS_gene_windows": n_tss_genes,
    "n_TSS_intergenic_windows": n_tss_negs,
    "mean_GRO_TTS_genes": mean_tts_genes,
    "mean_GRO_TTS_intergenic": mean_tts_negs,
    "n_TTS_gene_windows": n_tts_genes,
    "n_TTS_intergenic_windows": n_tts_negs,
})
profiles.to_csv(profiles_tsv, sep="\t", index=False)
print(f"Wrote: {profiles_tsv}")

# ------------------------
# Plot mean CPM-style profiles (absolute signal)
# ------------------------
plt.figure(figsize=(11, 4.5))
plt.subplot(1, 2, 1)
plt.plot(OFFSETS, mean_tss_genes, label="genes", linewidth=2)
plt.plot(OFFSETS, mean_tss_negs,  label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("GRO-seq around TSS (±500 bp)")
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Mean GRO-seq (normalized)")
plt.legend(frameon=False)

plt.subplot(1, 2, 2)
plt.plot(OFFSETS, mean_tts_genes, label="genes", linewidth=2)
plt.plot(OFFSETS, mean_tts_negs,  label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("GRO-seq around TTS (±500 bp)")
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Mean GRO-seq (normalized)")
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig(plot_mean_png, dpi=300)
plt.show()
print(f"Wrote: {plot_mean_png}")

# ------------------------
# Weighted density (shape × magnitude), with optional smoothing
# ------------------------
# Build probability densities
dens_tss_genes = to_prob_density(mean_tss_genes)
dens_tss_negs  = to_prob_density(mean_tss_negs)
dens_tts_genes = to_prob_density(mean_tts_genes)
dens_tts_negs  = to_prob_density(mean_tts_negs)

if SMOOTH_SD_BP > 0:
    dens_tss_genes = to_prob_density(gaussian_smooth(dens_tss_genes, sd_bp=SMOOTH_SD_BP))
    dens_tss_negs  = to_prob_density(gaussian_smooth(dens_tss_negs,  sd_bp=SMOOTH_SD_BP))
    dens_tts_genes = to_prob_density(gaussian_smooth(dens_tts_genes, sd_bp=SMOOTH_SD_BP))
    dens_tts_negs  = to_prob_density(gaussian_smooth(dens_tts_negs,  sd_bp=SMOOTH_SD_BP))

# Panel-wise masses (absolute signal)
mass_tss_genes = float(np.nansum(mean_tss_genes))
mass_tss_negs  = float(np.nansum(mean_tss_negs))
mass_tts_genes = float(np.nansum(mean_tts_genes))
mass_tts_negs  = float(np.nansum(mean_tts_negs))

# Apply weights (normalize weights within panel so largest mass == 1)
def weight_panel(d1, d2, m1, m2):
    mm = max(m1, m2, 1e-12)
    return d1 * (m1 / mm), d2 * (m2 / mm)

wd_tss_genes, wd_tss_negs = weight_panel(dens_tss_genes, dens_tss_negs, mass_tss_genes, mass_tss_negs)
wd_tts_genes, wd_tts_negs = weight_panel(dens_tts_genes, dens_tts_negs, mass_tts_genes, mass_tts_negs)

# Save weighted densities (raw, before display scaling)
wd_df = pd.DataFrame({
    "offset_bp": OFFSETS,
    "weighted_density_TSS_genes": wd_tss_genes,
    "weighted_density_TSS_intergenic": wd_tss_negs,
    "weighted_density_TTS_genes": wd_tts_genes,
    "weighted_density_TTS_intergenic": wd_tts_negs,
})
wd_df.to_csv(wd_tsv, sep="\t", index=False)
print(f"Wrote: {wd_tsv}")

# Display scaling (0–1 per panel across the two curves)
gmax_tss = max(wd_tss_genes.max(), wd_tss_negs.max(), 1e-12)
gmax_tts = max(wd_tts_genes.max(), wd_tts_negs.max(), 1e-12)

plt.figure(figsize=(11, 4.5))

plt.subplot(1, 2, 1)
plt.plot(OFFSETS, wd_tss_genes / gmax_tss, label="genes", linewidth=2)
plt.plot(OFFSETS, wd_tss_negs  / gmax_tss, label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Weighted density around TSS (±500 bp)")
plt.xlabel("Position relative to TSS (bp)")
plt.ylabel("Weighted density (0–1 across panel)")
plt.ylim(0, 1)
plt.legend(frameon=False)

plt.subplot(1, 2, 2)
plt.plot(OFFSETS, wd_tts_genes / gmax_tts, label="genes", linewidth=2)
plt.plot(OFFSETS, wd_tts_negs  / gmax_tts, label="intergenic", linewidth=2)
plt.axvline(0, linestyle="--", linewidth=1)
plt.title("Weighted density around TTS (±500 bp)")
plt.xlabel("Position relative to TTS (bp)")
plt.ylabel("Weighted density (0–1 across panel)")
plt.ylim(0, 1)
plt.legend(frameon=False)

plt.tight_layout()
plt.savefig(plot_wd_png, dpi=300)
plt.show()
print(f"Wrote: {plot_wd_png}")
